Importação das bibliotecas

In [ ]:
import pandas as pd
import os

Carregamento dos dados da camada Bronze

In [ ]:
# Definindo os caminhos para as camadas Bronze e Silver
bronze_path = 'raw'
silver_path = 'silver'

# Nome do arquivo de origem
raw_file = 'lancamentos-comerciais-por-distribuidoras.csv'

# Criando o caminho completo para o arquivo
file_path = os.path.join(bronze_path, raw_file)

# Carregando o dataset
try:
    df = pd.read_csv(file_path, sep=';', encoding='utf-8')
    print("Arquivo CSV bruto carregado com sucesso!")
    print(f"O DataFrame tem {df.shape[0]} linhas e {df.shape[1]} colunas.")
except FileNotFoundError:
    print(f"Erro: Arquivo '{file_path}' não encontrado. Certifique-se de que ele está na pasta 'bronze'.")
    df = None

# Visualizando as primeiras linhas do dataframe
if df is not None:
    display(df.head())

Dicionário de Dados e Renomeação das Colunas

In [ ]:
# Dicionário para renomear as colunas
rename_dict = {
    'DATA_LANCAMENTO_OBRA': 'data_lancamento',
    'TITULO_ORIGINAL': 'titulo_original',
    'CPB_ROE': 'cpb_roe',
    'TIPO_OBRA': 'tipo_obra',
    'PAIS_OBRA': 'pais_obra',
    'PUBLICO_TOTAL': 'publico_total',
    'RENDA_TOTAL': 'renda_total',
    'RAZAO_SOCIAL_DISTRIBUIDORA': 'distribuidora',
    'REGISTRO_DISTRIBUIDORA': 'registro_distribuidora',
    'CNPJ_DISTRIBUIDORA': 'cnpj_distribuidora'
}

df_renamed = df.rename(columns=rename_dict)

print("Colunas renomeadas com sucesso!")
display(df_renamed.head())

Processo de Limpeza e Transformação

In [ ]:
# Copiando o dataframe para evitar alterações no original durante o processo
df_processed = df_renamed.copy()

# 1. Tratamento da coluna 'renda_total'
# Remove 'R$ ', remove o ponto como separador de milhar e substitui a vírgula por ponto decimal
df_processed['renda_total'] = df_processed['renda_total'].replace({'R\$ ': '', '\.': ''}, regex=True).str.replace(',', '.')
df_processed['renda_total'] = pd.to_numeric(df_processed['renda_total'], errors='coerce')
df_processed['renda_total'] = df_processed['renda_total'].fillna(0) # Substitui valores nulos por 0

# 2. Tratamento da coluna 'publico_total'
df_processed['publico_total'] = pd.to_numeric(df_processed['publico_total'], errors='coerce').fillna(0).astype(int)

# 3. Tratamento da coluna 'data_lancamento'
df_processed['data_lancamento'] = pd.to_datetime(df_processed['data_lancamento'], format='%d/%m/%Y', errors='coerce')

# 4. Feature Engineering: Criando colunas de Ano, Mês e Dia
df_processed['ano_lancamento'] = df_processed['data_lancamento'].dt.year
df_processed['mes_lancamento'] = df_processed['data_lancamento'].dt.month
df_processed['dia_lancamento'] = df_processed['data_lancamento'].dt.day

# 5. Padronização de colunas de texto para minúsculas
text_columns = ['titulo_original', 'tipo_obra', 'pais_obra', 'distribuidora']
for col in text_columns:
    df_processed[col] = df_processed[col].str.lower()
    
# 6. Tratamento de valores ausentes em colunas de texto
string_columns = ['titulo_original', 'cpb_roe', 'tipo_obra', 'pais_obra', 'distribuidora', 'cnpj_distribuidora']
for col in string_columns:
    df_processed[col].fillna('Não informado', inplace=True)
    
# 7. Garantir o tipo de dado para a coluna de registro (tratando como string para segurança)
df_processed['registro_distribuidora'] = df_processed['registro_distribuidora'].astype(str)

print("Limpeza e transformação de dados concluídas.")
print("\nTipos de dados após a transformação:")
print(df_processed.info())

display(df_processed.head())